# One-GPU Colab GRPO pilot: Tier-5 Y wholesaler

This is the first actual RL run. It trains Qwen3-0.6B with a LoRA adapter using the native deterministic BeerEpisode reward on development seeds 0–2 only. The local trainer converts a strict JSON quantity into the same integer `place_order` action. The pilot tests the RL signal; a selected checkpoint still needs a separate native Verifiers tool-call evaluation.

Use a GPU runtime. Do not paste the Akash key here; this pilot does not call Akash. Before running, make sure `REF` points to a pushed commit containing this notebook and `scripts/train_colab_grpo_wholesaler.py`.

In [ ]:
import os, pathlib, subprocess, sys

REPO_URL = "https://github.com/ConstantinVictorBeatErtel/beer_distribution_RL.git"
REF = "main"  # replace with the exact audited commit once pushed
REPO_DIR = pathlib.Path("/content/beer_distribution_RL")
if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REF])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", "-B", REF, "FETCH_HEAD"])
os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

In [ ]:
# Install the exact environment dependency plus the one-GPU training stack.
%pip install -q "verifiers==0.2.0" "transformers>=4.51,<6" "accelerate>=1.2" "peft>=0.14" "bitsandbytes>=0.45" safetensors
%pip install -q -e environments/beer_distribution_game --no-deps
subprocess.check_call([sys.executable, "-m", "py_compile", "scripts/train_colab_grpo_wholesaler.py"])
print("install and compile OK")

In [ ]:
# Preflight: confirm the GPU and that training constructs development rows only.
subprocess.run(["nvidia-smi"], check=False)
subprocess.check_call([
    sys.executable, "scripts/train_colab_grpo_wholesaler.py",
    "--dry-run", "--output-dir", "/content/beer_grpo_dryrun",
])

In [ ]:
# Run the 10-update development smoke. Checkpoint and logs go to Drive.
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/beer_distribution_rl_artifacts/beer_wholesaler_qwen3_0p6b_grpo_smoke"
RUN_ENV = {**os.environ, "PYTHONUNBUFFERED": "1"}
subprocess.check_call([
    sys.executable, "scripts/train_colab_grpo_wholesaler.py",
    "--updates", "10",
    "--group-size", "8",
    "--train-seeds", "0", "1", "2",
    "--train-minibatch", "2",
    "--inference-minibatch", "8",
    "--max-new-tokens", "16",
    "--output-dir", OUT,
], env=RUN_ENV)

In [ ]:
# Held-out validation: only run after selecting the smoke checkpoint.
# This constructs validation seeds 0–4 and never feeds their rewards back into training.
subprocess.check_call([
    sys.executable, "scripts/train_colab_grpo_wholesaler.py",
    "--eval-only",
    "--adapter", OUT + "/adapter",
    "--eval-split", "validation",
    "--eval-seeds", "0", "1", "2", "3", "4",
    "--tier5-controls",
    "--output-dir", OUT + "/validation_eval",
])